# Clasificación multimodal de memes con odio: entrenamiento en Colab

Este notebook corre el pipeline completo del repositorio con GPU:

1. Extrae los embeddings de imagen (ResNet50) y de texto (DistilBERT) una sola vez
2. Entrena tres clasificadores: solo imagen, solo texto y fusión multimodal
3. Evalúa en el split dev y genera métricas y gráficas

**Antes de empezar:** Entorno de ejecución > Cambiar tipo de entorno > GPU (T4).

**Datos:** sube `memes_colab.zip` a la raíz de tu Google Drive (o súbelo directo a Colab en la celda 2).

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/Ivifer1/meme_classification_project.git
%cd meme_classification_project
!pip install -q -r requirements.txt

## 1. Datos
Opción A: desde Google Drive (recomendado). Opción B: si no está en Drive, se abre un botón para subir el zip.

In [ ]:
import os
ZIP = '/content/drive/MyDrive/memes_colab.zip'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(e)

if not os.path.exists(ZIP):
    from google.colab import files
    print('No está en Drive, sube memes_colab.zip:')
    up = files.upload()
    ZIP = '/content/' + list(up)[0]

!mkdir -p /content/memes && unzip -q -o "$ZIP" -d /content/memes
os.environ['MEME_DATA_DIR'] = '/content/memes'
!ls /content/memes && ls /content/memes/img | wc -l

## 2. Extracción de características
ResNet50 y DistilBERT están congelados, así que sus embeddings se calculan una sola vez y se guardan. Con GPU tarda unos minutos.

In [ ]:
!python -m src.extract_features --splits train dev

## 3. Entrenamiento
Se entrena el clasificador de cada modalidad por separado y el multimodal, con early stopping según el ROC-AUC de validación.

In [ ]:
!python -m src.train

## 4. Evaluación en dev

In [ ]:
!python -m src.evaluate
from IPython.display import Image, display, Markdown
display(Markdown(open('results/metrics.md').read()))
display(Image('results/roc_curves.png', width=480))
display(Image('results/confusion_matrix_fusion.png', width=340))

## 5. Descargar resultados
Descarga el zip y copia la carpeta `results/` dentro del proyecto en tu computadora.

In [ ]:
!zip -q -r /content/resultados.zip results
from google.colab import files
files.download('/content/resultados.zip')